In [1]:
import os, re, sys, gc, json, time, hashlib, shutil, subprocess, unicodedata
from pathlib import Path
!pip install ijson
import ijson
import requests
import sentencepiece as spm


OUTPUT_DIR = Path("/kaggle/working/telugu_tokenizer_v3")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BUCKET_DIR = OUTPUT_DIR / "buckets"

TRAIN_PATH = OUTPUT_DIR / "telugu_train.txt"
VAL_PATH = OUTPUT_DIR / "telugu_val.txt"
TEST_PATH = OUTPUT_DIR / "telugu_test.txt"

MODEL_PREFIX = OUTPUT_DIR / "telugu_sp"
MODEL_PATH = Path(f"{MODEL_PREFIX}.model")
VOCAB_PATH = Path(f"{MODEL_PREFIX}.vocab")

N_BUCKETS = 256


KAGGLE_SOURCES = [
    {
        "path": "/kaggle/input/datasets/tanoojtadepalli/bashaversedatasrt/bhashaverse_telugu.json",
        "field": "text",
        "extensions": (".json",)
    },
    {
        "path": "/kaggle/input/datasets/tanoojtadepalli/podcsast/telugu_podcast.json",
        "field": "text",
        "extensions": (".json",)
    },
    {
        "path": "/kaggle/input/datasets/tanoojtadepalli/scrapedclean/scraped_cleaned.jsonl",
        "field": "text",
        "extensions": (".jsonl",)
    }
]


INDICCORP_REPO = "ai4bharat/IndicCorpV2"
INDICCORP_SPLIT = "tel_Telu"
print("config loaded ")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.8/149.8 kB 7.4 MB/s eta 0:00:00
config loaded 


## Checking if the files exists or not 

In [2]:
from pathlib import Path

for src in KAGGLE_SOURCES:
    root = Path(src["path"])

    print("\nSOURCE:", root)
    print("Exists:", root.exists())

    if root.exists():
        files = list(root.rglob("*"))

        print("Total files:", sum(p.is_file() for p in files))

        for p in files[:20]:
            print(" ", p)


SOURCE: /kaggle/input/datasets/tanoojtadepalli/bashaversedatasrt/bhashaverse_telugu.json
Exists: True
Total files: 0

SOURCE: /kaggle/input/datasets/tanoojtadepalli/podcsast/telugu_podcast.json
Exists: True
Total files: 0

SOURCE: /kaggle/input/datasets/tanoojtadepalli/scrapedclean/scraped_cleaned.jsonl
Exists: True
Total files: 0


## Cleaning the telugu corpus 
#### 1.Removing the HTML Tags
#### 2.Remove URLS
#### 3. Unicode Normalization 
#### 4.Keeping only telugu characters and removing all non telugu characters 
#### 5.Replacing mutiple spaces and tabs to single space 
#### 6.Multiple consecutive new lines are converted to a single new line 
#### 7.Adding spacing around Punctuation
#### 8.Since the punctuation steps can introduce some more extra spaces cleaning the extra spaces  


In [3]:
html_re  = re.compile(r'<[^>]*>')
url_re   = re.compile(r'http\S+|www\.\S+', re.IGNORECASE)
keep_re  = re.compile(r'[^\u0C00-\u0C7F\u200D\u200C\s.,!?0-9]')   # Telugu block
punct_re = re.compile(r'([.,!?।॥])')
ws_re    = re.compile(r'\s+')
nl_re    = re.compile(r'\n+')
tab_re   = re.compile(r'[ \t]+')

def clean_text(text):
    text = html_re.sub('', text)
    text = url_re.sub('', text)
    text = unicodedata.normalize('NFC', text)
    text = keep_re.sub('', text)
    text = tab_re.sub(' ', text)
    text = nl_re.sub('\n', text).strip()
    text = punct_re.sub(r' \1 ', text)
    text = tab_re.sub(' ', text).strip()
    return text

## Splitting the large telugu manual data into small 256 buckets in order to avoid the memory excedded isue 
##### 1.Creates a clean bucket directory -If the directory already exisits the directory is deleted and a clean directory of buckets is created 
##### 2.Reads each dataset source if the mime type is jsonl then it reads using json and if the mime type is json then it reads using ijson 
##### 3.Extracts and cleans the text .It splits the text into individual lines/sentences and also adds a "M" tag before each sentence 
##### 4.Uses A MD5 hash to chose the bucket .The first 2 hexadecimal characters give a number between 0-255.That number determines which bucket recieves the sentence 

In [4]:
import hashlib
import os
import json
import ijson
import shutil

BUCKET_DIR = "/kaggle/working/telugu_buckets"
N_BUCKETS = 256

# Start clean if notebook is being rerun
if os.path.exists(BUCKET_DIR):
    shutil.rmtree(BUCKET_DIR)

os.makedirs(BUCKET_DIR)

# Open bucket files
manual_files = [
    open(
        os.path.join(BUCKET_DIR, f"bucket_{i:03d}.txt"),
        "w",
        encoding="utf-8"
    )
    for i in range(N_BUCKETS)
]

manual_candidates = 0

for src in KAGGLE_SOURCES:

    path = src["path"]

    print("Reading:", path)

    if path.endswith(".jsonl"):

        with open(path, "r", encoding="utf-8") as f:

            for raw in f:

                try:
                    record = json.loads(raw)
                except:
                    continue

                text = record.get("text")

                if not isinstance(text, str):
                    continue

                text = clean_text(text)

                for line in text.split("\n"):

                    line = line.strip()

                    if not line:
                        continue

                    h = int(
                        hashlib.md5(
                            line.encode("utf-8")
                        ).hexdigest()[:2],
                        16
                    )

                    manual_files[h].write(
                        "M\t" + line + "\n"
                    )

                    manual_candidates += 1

    else:

        with open(path, "rb") as f:

            for record in ijson.items(f, "item"):

                text = record.get("text")

                if not isinstance(text, str):
                    continue

                text = clean_text(text)

                for line in text.split("\n"):

                    line = line.strip()

                    if not line:
                        continue

                    h = int(
                        hashlib.md5(
                            line.encode("utf-8")
                        ).hexdigest()[:2],
                        16
                    )

                    manual_files[h].write(
                        "M\t" + line + "\n"
                    )

                    manual_candidates += 1

for f in manual_files:
    f.close()

print(
    manual_candidates,
    "manual sentence candidates written to disk"
)

Reading: /kaggle/input/datasets/tanoojtadepalli/bashaversedatasrt/bhashaverse_telugu.json
Reading: /kaggle/input/datasets/tanoojtadepalli/podcsast/telugu_podcast.json
Reading: /kaggle/input/datasets/tanoojtadepalli/scrapedclean/scraped_cleaned.jsonl
23272604 manual sentence candidates written to disk


##  Loading the Indicorp dataset 
#### 1.First reopens the already exisiting 256 buckets 
#### 2.Streams Indicorp instead of downloading everything 
#### 3.cleans and split each sentence.For every sentnece it adds the "D" tag before evry sentence 
#### 4.hashes and keeps into the bucket .This is process is repeated until we hit the 8GB data limit 


In [5]:
def bucket_of(line):
    return int(
        hashlib.md5(
            line.encode("utf-8")
        ).hexdigest()[:2],
        16
    )

In [6]:
from datasets import load_dataset

MAX_BYTES = 8 * 1024**3

# Re-open buckets in append mode
bucket_files = [
    open(
        os.path.join(BUCKET_DIR, f"bucket_{i:03d}.txt"),
        "a",
        encoding="utf-8"
    )
    for i in range(N_BUCKETS)
]

print(
    f"streaming {INDICCORP_REPO} "
    f"split={INDICCORP_SPLIT} ..."
)

ds = load_dataset(
    INDICCORP_REPO,
    "indiccorp_v2",
    split=INDICCORP_SPLIT,
    streaming=True
)

downloaded_candidates = 0
downloaded_docs = 0
written_bytes = 0

t0 = time.time()

for i, ex in enumerate(ds, 1):

    text = ex.get("text")

    if not text:
        continue

    cleaned = clean_text(text)

    if not cleaned:
        continue

    for line in re.split(r"[.।॥\n!?]", cleaned):

        line = line.strip()

        if not line:
            continue

        data = "D\t" + line + "\n"
        data_bytes = len(data.encode("utf-8"))

        # Stop after 8 GB
        if written_bytes + data_bytes > MAX_BYTES:
            break

        b = bucket_of(line)

        bucket_files[b].write(data)

        written_bytes += data_bytes
        downloaded_candidates += 1

    downloaded_docs += 1

    # Print every 10,000 documents
    if i % 10_000 == 0:

        print(
            f"  {i:,} docs | "
            f"{downloaded_candidates:,} candidates | "
            f"{written_bytes / (1024**3):.2f} GB written | "
            f"{time.time() - t0:.0f}s"
        )

    if written_bytes >= MAX_BYTES:
        print("\nReached 8 GB limit. Stopping IndicCorp.")
        break


# Close bucket files
for f in bucket_files:
    f.close()


# Final statistics
print(
    f"\n[indiccorp] "
    f"{downloaded_candidates:,} candidates"
)

print(
    f"[indiccorp] "
    f"{downloaded_docs:,} documents processed"
)

print(
    f"[indiccorp] "
    f"{written_bytes / (1024**3):.2f} GB written"
)

print(
    f"[indiccorp] elapsed: "
    f"{(time.time() - t0) / 60:.1f} min"
)


# Remaining disk
_, _, free = shutil.disk_usage("/kaggle/working")

print(
    f"[disk] free: "
    f"{free / (1024**3):.2f} GB"
)

streaming ai4bharat/IndicCorpV2 split=tel_Telu ...


README.md: 0.00B [00:00, ?B/s]


Reached 8 GB limit. Stopping IndicCorp.

[indiccorp] 46,190,318 candidates
[indiccorp] 11,673,309 documents processed
[indiccorp] 8.00 GB written
[indiccorp] elapsed: 26.1 min
[disk] free: 6.11 GB


## Removing the duplicates stage 
#### 1.creates the final output files for train data,test data and validation data 
#### 2.Performms deduplication of each bucket using a set 
#### 3.splits unigque sentence to train validation and test files and write data into themn 
#### 4. And also provides the statistics fir the number of uniqie sentences and words in btoh manual and downloaded splits 

In [7]:
train_out = open(TRAIN_PATH, "w", encoding="utf-8")
val_out   = open(VAL_PATH,   "w", encoding="utf-8")
test_out  = open(TEST_PATH,  "w", encoding="utf-8")

unique_total = duplicate_total = 0
manual_unique = downloaded_unique = 0
manual_words  = downloaded_words  = 0

t0 = time.time()

for b in range(N_BUCKETS):

    bucket_path = os.path.join(
        BUCKET_DIR,
        f"bucket_{b:03d}.txt"
    )

    if not os.path.exists(bucket_path):
        continue

    seen = {}

    with open(bucket_path, "r", encoding="utf-8") as f:

        for raw in f:

            try:
                source, line = raw.rstrip("\n").split("\t", 1)
            except ValueError:
                continue

            if line in seen:
                duplicate_total += 1
                continue

            seen[line] = source
            unique_total += 1

            if source == "M":
                manual_unique += 1
                manual_words += len(line.split())
            else:
                downloaded_unique += 1
                downloaded_words += len(line.split())

            # Deterministic 80/10/10 split by hash
            split_value = (
                int(
                    hashlib.md5(
                        line.encode("utf-8")
                    ).hexdigest(),
                    16
                ) % 100
            )

            if split_value < 80:
                train_out.write(line + "\n")

            elif split_value < 90:
                val_out.write(line + "\n")

            else:
                test_out.write(line + "\n")

    # Delete bucket after processing
    os.remove(bucket_path)

    del seen

    if (b + 1) % 16 == 0:

        _, _, free = shutil.disk_usage(
            "/kaggle/working"
        )

        print(
            f"  bucket {b+1}/{N_BUCKETS}  "
            f"unique={unique_total:,}  "
            f"free={free / (1024**3):.2f} GB  "
            f"({time.time()-t0:.0f}s)"
        )


train_out.close()
val_out.close()
test_out.close()


# Clean up empty bucket directory
if os.path.exists(BUCKET_DIR):

    try:
        os.rmdir(BUCKET_DIR)
    except OSError:
        pass


print(
    f"\n[dedup] DONE.  "
    f"elapsed: {(time.time()-t0)/60:.1f} min"
)

print(
    f"  unique total:    {unique_total:,}"
)

print(
    f"  duplicates:      {duplicate_total:,}"
)

print(
    f"  manual unique:   {manual_unique:,}  "
    f"({manual_words:,} words)"
)

print(
    f"  downloaded uniq: {downloaded_unique:,}  "
    f"({downloaded_words:,} words)"
)


_, _, free = shutil.disk_usage(
    "/kaggle/working"
)

print(
    f"\n[disk] free: "
    f"{free / (1024**3):.2f} GB"
)

  bucket 16/256  unique=3,797,169  free=6.17 GB  (32s)
  bucket 32/256  unique=7,593,533  free=6.23 GB  (66s)
  bucket 48/256  unique=11,391,522  free=6.29 GB  (99s)
  bucket 64/256  unique=15,186,285  free=6.36 GB  (132s)
  bucket 80/256  unique=18,982,820  free=6.42 GB  (164s)
  bucket 96/256  unique=22,779,864  free=6.48 GB  (196s)
  bucket 112/256  unique=26,578,130  free=6.54 GB  (231s)
  bucket 128/256  unique=30,368,934  free=6.61 GB  (263s)
  bucket 144/256  unique=34,162,562  free=6.68 GB  (296s)
  bucket 160/256  unique=37,959,200  free=6.75 GB  (328s)
  bucket 176/256  unique=41,751,194  free=6.82 GB  (362s)
  bucket 192/256  unique=45,547,394  free=6.89 GB  (395s)
  bucket 208/256  unique=49,342,243  free=6.95 GB  (427s)
  bucket 224/256  unique=53,134,828  free=7.02 GB  (458s)
  bucket 240/256  unique=56,931,864  free=7.08 GB  (490s)
  bucket 256/256  unique=60,727,122  free=7.14 GB  (521s)

[dedup] DONE.  elapsed: 8.7 min
  unique total:    60,727,122
  duplicates:      8

## Count the numbr of lines in each train,test, and validation sets

In [8]:
def count_lines(path):
    r = subprocess.run(
        ["wc", "-l", str(path)],
        capture_output=True,
        text=True
    )
    return int(r.stdout.split()[0])


n_train = count_lines(TRAIN_PATH)
n_val   = count_lines(VAL_PATH)
n_test  = count_lines(TEST_PATH)

n_total = n_train + n_val + n_test


print(
    f"Train:  {n_train:>12,}  "
    f"({n_train/n_total*100:.2f}%)   "
    f"{TRAIN_PATH.stat().st_size / (1024**3):.2f} GB"
)

print(
    f"Val:    {n_val:>12,}  "
    f"({n_val/n_total*100:.2f}%)   "
    f"{VAL_PATH.stat().st_size / (1024**3):.2f} GB"
)

print(
    f"Test:   {n_test:>12,}  "
    f"({n_test/n_total*100:.2f}%)   "
    f"{TEST_PATH.stat().st_size / (1024**3):.2f} GB"
)

print(
    f"TOTAL:  {n_total:>12,}"
)

Train:    48,581,452  (80.00%)   9.89 GB
Val:       6,069,130  (9.99%)   1.24 GB
Test:      6,076,540  (10.01%)   1.24 GB
TOTAL:    60,727,122


## Sentence piece tokenizer stage 
#### 1.counts the training data and splits into two vocabs 5k and 10k 
#### 2.trains the sentence piece tokenizer and creates the correspnding .vocabn and .model files 

In [9]:
with open(TRAIN_PATH, encoding="utf-8") as f:
    n_lines = sum(1 for _ in f)

print(f"{n_lines:,} lines in {TRAIN_PATH.name}")
print(
    f"size: "
    f"{TRAIN_PATH.stat().st_size / (1024**3):.2f} GB\n"
)


# Train both 16K and 32K vocabularies
VOCAB_SIZES = [5_000, 10_000]


for VOCAB_SIZE in VOCAB_SIZES:

    print(f"\n{'='*60}")
    print(f"Training vocab_size={VOCAB_SIZE:,}")
    print(f"{'='*60}")

    t0 = time.time()

    model_prefix = OUTPUT_DIR / f"telugu_sp_{VOCAB_SIZE//1000}k"

    spm.SentencePieceTrainer.train(
        input=str(TRAIN_PATH),
        model_prefix=str(model_prefix),

        vocab_size=VOCAB_SIZE,

        model_type="unigram",

        character_coverage=1.0,

        normalization_rule_name="identity",

        byte_fallback=True,

        pad_id=0,
        unk_id=1,
        bos_id=2,
        eos_id=3,

        pad_piece="<pad>",
        unk_piece="<unk>",
        bos_piece="<s>",
        eos_piece="</s>",

        # Same settings as your Nepali tokenizer
        shuffle_input_sentence=False,
        input_sentence_size=200_000,
        train_extremely_large_corpus=True,
        num_threads=2,
    )

    print(
        f"[train] {VOCAB_SIZE:,} DONE "
        f"in {(time.time()-t0)/60:.1f} min"
    )

    model_path = Path(f"{model_prefix}.model")
    vocab_path = Path(f"{model_prefix}.vocab")

    print(
        f"[model] {model_path.name}  "
        f"{model_path.stat().st_size / (1024**2):.2f} MB"
    )

    print(
        f"[vocab] {vocab_path.name}  "
        f"{vocab_path.stat().st_size / (1024**2):.2f} MB"
    )

48,581,452 lines in telugu_train.txt
size: 9.89 GB


Training vocab_size=5,000
[train] 5,000 DONE in 1.0 min
[model] telugu_sp_5k.model  0.13 MB
[vocab] telugu_sp_5k.vocab  0.13 MB

Training vocab_size=10,000
[train] 10,000 DONE in 1.0 min
[model] telugu_sp_10k.model  0.28 MB
[vocab] telugu_sp_10k.vocab  0.28 MB


sentencepiece_trainer.cc(78) LOG(INFO) Starts training with : 
trainer_spec {
  input: /kaggle/working/telugu_tokenizer_v3/telugu_train.txt
  input_format: 
  model_prefix: /kaggle/working/telugu_tokenizer_v3/telugu_sp_5k
  model_type: UNIGRAM
  vocab_size: 5000
  self_test_sample_size: 0
  character_coverage: 1
  input_sentence_size: 200000
  shuffle_input_sentence: 0
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 2
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  required_chars: 
  byte_fallback: 1
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 1
  seed_sentencepieces_file: 
  hard_vocab_limit: 1
  use_all_vocab: 0
  unk_id: 1
  bos_id: 2
  eos_id: 3
  pad_id: 0
  unk_piece: <unk>
  bos_piece: <s>
  eos_piece: </s>


## testing both toeknizers on the held out validataion datset 
#### we chose the best tokenizer on the basisi of the fertility rate and the unk tokens rates 
#### for 5k vocab we got aa fertilikty rate of 1.67 and fopr 10 k vocab we got a fertitlitty rate of  1.44 clearing shpwing 10k is the winner here 

In [10]:
import shutil

MAX_VAL_LINES = 5000


def load_validation_sample(path, n=5000):
    texts = []

    with open(path, encoding="utf-8") as f:

        for line in f:

            line = line.strip()

            if not line:
                continue

            texts.append(line)

            if len(texts) >= n:
                break

    return texts


val_texts = load_validation_sample(
    VAL_PATH,
    MAX_VAL_LINES
)

print(
    "Validation lines used:",
    len(val_texts)
)


def quick_stats(sp, texts, n=5000, print_every=1000):

    n_tok = 0
    n_word = 0
    n_unk = 0

    unk_id = sp.unk_id()

    for i, line in enumerate(texts[:n], 1):

        ids = sp.encode(
            line,
            out_type=int
        )

        n_tok += len(ids)

        n_word += len(line.split())

        n_unk += sum(
            1 for t in ids
            if t == unk_id
        )

        if i % print_every == 0:
            print(
                " ",
                i,
                "lines tokenized"
            )

    return {
        "fertility": n_tok / max(n_word, 1),
        "unk_rate": n_unk / max(n_tok, 1)
    }


# Compare 16K and 32K
vocab_sizes = [5_000, 10_000]

results = {}

for v in vocab_sizes:

    key = v

    print(
        f"\nvocab_size={v:,}"
    )

    model_path = (
        OUTPUT_DIR /
        f"telugu_sp_{v//1000}k.model"
    )

    sp_v = spm.SentencePieceProcessor(
        model_file=str(model_path)
    )

    results[key] = quick_stats(
        sp_v,
        val_texts
    )

    print(
        " ",
        results[key]
    )


# Select the best model
# Lowest UNK rate first,
# fertility used as tie-breaker.

best_vocab = min(
    results,
    key=lambda v: (
        results[v]["unk_rate"],
        results[v]["fertility"]
    )
)

print(
    f"\npicking vocab_size={best_vocab:,} "
    f"(lowest UNK rate)"
)


# Copy selected model to standard names
best_model = (
    OUTPUT_DIR /
    f"telugu_sp_{best_vocab//1000}k.model"
)

best_vocab_file = (
    OUTPUT_DIR /
    f"telugu_sp_{best_vocab//1000}k.vocab"
)

shutil.copy(
    best_model,
    OUTPUT_DIR / "telugu_sp.model"
)

shutil.copy(
    best_vocab_file,
    OUTPUT_DIR / "telugu_sp.vocab"
)


# Load selected tokenizer
MODEL_PATH = OUTPUT_DIR / "telugu_sp.model"
VOCAB_PATH = OUTPUT_DIR / "telugu_sp.vocab"

sp = spm.SentencePieceProcessor(
    model_file=str(MODEL_PATH)
)

print(
    f"\nSelected tokenizer vocab size: "
    f"{sp.get_piece_size():,}"
)

Validation lines used: 5000

vocab_size=5,000
  1000 lines tokenized
  2000 lines tokenized
  3000 lines tokenized
  4000 lines tokenized
  5000 lines tokenized
  {'fertility': 1.6750567787225257, 'unk_rate': 0.0}

vocab_size=10,000
  1000 lines tokenized
  2000 lines tokenized
  3000 lines tokenized
  4000 lines tokenized
  5000 lines tokenized
  {'fertility': 1.441447528952456, 'unk_rate': 0.0}

picking vocab_size=10,000 (lowest UNK rate)

Selected tokenizer vocab size: 10,000


####  Saving the best model to the directory 

In [11]:
import os, shutil, json

out_root = (
    "/kaggle/working"
    if os.path.exists("/kaggle/working")
    else "."
)

out_dir = os.path.join(
    out_root,
    "telugu/tokenizer"
)

os.makedirs(
    out_dir,
    exist_ok=True
)

for fname in [
    "telugu_sp.model",
    "telugu_sp.vocab"
]:

    shutil.copy(
        os.path.join(OUTPUT_DIR, fname),
        os.path.join(
            out_dir,
            fname
        )
    )

vocab = [
    {
        "id": i,
        "piece": sp.id_to_piece(i),
        "score": sp.get_score(i)
    }

    for i in range(
        sp.vocab_size()
    )
]

with open(
    os.path.join(
        out_dir,
        "vocab.json"
    ),
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        vocab,
        f,
        ensure_ascii=False,
        indent=2
    )

print(
    "saved to",
    out_dir
)

print(
    os.listdir(out_dir)
)

saved to /kaggle/working/telugu/tokenizer
['telugu_sp.model', 'vocab.json', 'telugu_sp.vocab']


##### Loading the tokenizer and tokenizigin the train corpus 
##### the final tokens are 863 million from 48 miliion lines 

In [12]:
import numpy as np
import sentencepiece as spm

sp = spm.SentencePieceProcessor(
    model_file="/kaggle/working/telugu/tokenizer/telugu_sp.model"
)

TRAIN_PATH = "/kaggle/working/telugu_tokenizer_v3/telugu_train.txt"
OUT_PATH = "/kaggle/working/telugu_tokenizer_v3/telugu_train.bin"

CHUNK_SIZE = 500_000  # lines per chunk

vocab_size = sp.get_piece_size()

dtype = (
    np.uint16
    if vocab_size < 65536
    else np.uint32
)

total_tokens = 0
total_lines = 0
chunk = []

with open(
    TRAIN_PATH,
    encoding="utf-8"
) as f_in, open(
    OUT_PATH,
    "wb"
) as f_out:

    for line in f_in:

        line = line.strip()

        if not line:
            continue

        ids = sp.encode(
            line,
            out_type=int
        )

        chunk.extend(ids)

        total_tokens += len(ids)
        total_lines += 1

        if total_lines % CHUNK_SIZE == 0:

            np.array(
                chunk,
                dtype=dtype
            ).tofile(f_out)

            chunk = []

            print(
                f"  {total_lines:,} lines processed, "
                f"{total_tokens:,} tokens so far"
            )

    if chunk:
        np.array(
            chunk,
            dtype=dtype
        ).tofile(f_out)


print(
    f"\nFinal: {total_lines:,} lines, "
    f"{total_tokens:,} tokens in {TRAIN_PATH}"
)

print(
    f"Saved to {OUT_PATH}, "
    f"dtype={dtype.__name__}"
)

  500,000 lines processed, 8,906,550 tokens so far
  1,000,000 lines processed, 17,525,026 tokens so far
  1,500,000 lines processed, 26,514,538 tokens so far
  2,000,000 lines processed, 35,463,375 tokens so far
  2,500,000 lines processed, 44,153,653 tokens so far
  3,000,000 lines processed, 53,153,030 tokens so far
  3,500,000 lines processed, 62,142,287 tokens so far
  4,000,000 lines processed, 70,876,040 tokens so far
  4,500,000 lines processed, 79,891,514 tokens so far
  5,000,000 lines processed, 88,496,477 tokens so far
  5,500,000 lines processed, 97,448,128 tokens so far
  6,000,000 lines processed, 106,436,383 tokens so far
  6,500,000 lines processed, 115,146,122 tokens so far
  7,000,000 lines processed, 124,126,406 tokens so far
  7,500,000 lines processed, 133,124,326 tokens so far
  8,000,000 lines processed, 141,807,572 tokens so far
  8,500,000 lines processed, 150,720,918 tokens so far
  9,000,000 lines processed, 159,708,333 tokens so far
  9,500,000 lines proces

In [5]:
import sentencepiece as spm
sp = spm.SentencePieceProcessor(model_file="C:/Users/Tanooj/Downloads/2025201068_repo/telugu/tokenizer/telugu_sp.model")
test_words = [
    "ఇది ఒక పరీక్షా వాక్యం.",
    "అంతర్జాతీయీకరణ ప్రపంచవ్యాప్తంగా అనేక మార్పులను తీసుకువచ్చింది.",
    "కృత్రిమ మేధస్సు భవిష్యత్తులో అనేక రంగాలను మార్చగలదు.",
]

for t in test_words:
    print(f"\ntext: {t}")
    print("pieces:", sp.encode(t, out_type=str))


text: ఇది ఒక పరీక్షా వాక్యం.
pieces: ['▁ఇది', '▁ఒక', '▁పరీక్ష', 'ా', '▁వాక్య', 'ం', '.']

text: అంతర్జాతీయీకరణ ప్రపంచవ్యాప్తంగా అనేక మార్పులను తీసుకువచ్చింది.
pieces: ['▁అంతర్జాతీయ', 'ీకరణ', '▁ప్రపంచవ్యాప్త', 'ంగా', '▁అనేక', '▁మార్పుల', 'ను', '▁తీసుకువచ్చ', 'ింది', '.']

text: కృత్రిమ మేధస్సు భవిష్యత్తులో అనేక రంగాలను మార్చగలదు.
pieces: ['▁కృత్రిమ', '▁మే', 'ధ', 'స్సు', '▁భవిష్యత్తులో', '▁అనేక', '▁రంగాల', 'ను', '▁మార్చ', 'గలదు', '.']


In [4]:
!pip install sentencepiece

   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   ---------------------------------------- 1.2/1.2 MB 13.5 MB/s  0:00:00
